# PCA
## Completed by William Brannock (svv8fs)

This notebook does the PCA tasks described in the final project notebook. I used the class notebook https://www.kaggle.com/code/ontoligent/uva-ds-5001-m07-02-pca-with-skl#Run-PCA as a guide. 

# Set Up

## Import

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from sklearn.decomposition import PCA

pio.renderers.default = "plotly_mimetype+notebook_connected"

## Config

In [2]:
data_home = "."
output_dir = "output"
data_prefix = "constitutions"
source_dir = output_dir
CSV_DELIM = "|"

# Set Sttings
bag = "CONSTITUTIONS"
n_components = 5

# Prepare the Data

## Import Tables

In [3]:
LIB = pd.read_csv(f"{source_dir}/{data_prefix}-LIB.csv", sep=CSV_DELIM).set_index("constitution_id")
TFIDF_L2 = pd.read_csv(f"{source_dir}/{data_prefix}-TFIDF_L2-{bag}.csv", sep=CSV_DELIM).set_index("constitution_id")
LIB = LIB.reindex(TFIDF_L2.index)

In [4]:
LIB.head()

,source_file_path,country,file_year,title,original_year,revision_year,provision_regex,constitution_len,n_provisions,n_chars
constitution_id,,,,,,,,,,
Afghanistan_2004,./data/Afghanistan_2004.txt,Afghanistan,2004,Afghanistan,2004.0,NaN,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,10435,175,66806
Albania_2008,./data/Albania_2008.txt,Albania,2008,Albania,1998.0,2008.0,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,13896,623,86022
Algeria_2008,./data/Algeria_2008.txt,Algeria,2008,Algeria,1963.0,2008.0,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,10605,197,66590
Andorra_1993,./data/Andorra_1993.txt,Andorra,1993,Andorra,1993.0,NaN,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,8722,295,55687
Angola_2010,./data/Angola_2010.txt,Angola,2010,Angola,2010.0,NaN,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,26657,856,175148


# Run PCA

In [5]:
pca_engine = PCA(n_components=n_components)
DCM = pd.DataFrame(pca_engine.fit_transform(TFIDF_L2), index=TFIDF_L2.index)
DCM.columns = ["PC{}".format(i) for i in DCM.columns]
DCM.head()

,PC0,PC1,PC2,PC3,PC4
constitution_id,,,,,
Afghanistan_2004,-0.000069,-0.000198,-0.000229,-0.000228,0.000982
Albania_2008,0.000389,-0.000481,0.000060,-0.000410,-0.000036
Algeria_2008,0.000420,-0.000521,0.000058,-0.000285,0.000027
Andorra_1993,0.000368,-0.000422,0.000121,-0.000375,-0.000067
Angola_2010,0.000365,-0.000438,0.000101,-0.000425,-0.000101


In [6]:
COMPS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_), index=TFIDF_L2.columns)
COMPS.columns = ["PC{}".format(i) for i in COMPS.columns]
COMPS.index.name = "term_str"
COMPS.head()

,PC0,PC1,PC2,PC3,PC4
term_str,,,,,
assent,-0.000021,0.000018,-5.383391e-06,-0.000002,-1.213410e-05
occasion,-0.000011,0.000009,-3.677951e-06,0.000001,-1.255053e-07
168,0.000001,-0.000003,1.917184e-07,-0.000002,-4.103434e-07
class,-0.000012,0.000004,-4.998969e-06,0.000004,-5.973868e-06
read,-0.000003,0.000003,-1.229196e-07,-0.000002,-6.693704e-06


# Inspect Components

In [7]:
def top_terms_for_pc(pc_num, n=10):
    pc = f"PC{pc_num}"
    positive = COMPS[pc].sort_values(ascending=False).head(n).rename("positive")
    negative = COMPS[pc].sort_values().head(n).rename("negative")
    return pd.concat([positive.reset_index(), negative.reset_index()], axis=1)


top_terms_for_pc(0)

,term_str,positive,term_str,negative
0,art,0.000666,subsection,-0.000610
1,congress,0.000140,governor,-0.000327
2,chamber,0.000075,house,-0.000305
3,deputies,0.000052,ad,-0.000168
4,down,0.000039,speaker,-0.000156
5,communal,0.000036,senate,-0.000122
6,autonomous,0.000035,advice,-0.000118
7,peoples,0.000032,officer,-0.000094
8,municipalities,0.000031,references,-0.000089
9,statute,0.000030,director,-0.000089


In [8]:
top_terms_for_pc(1)

,term_str,positive,term_str,negative
0,art,0.000870,congress,-0.000221
1,subsection,0.000378,peoples,-0.000054
2,governor,0.000213,chamber,-0.000047
3,house,0.000189,deputies,-0.000047
4,ad,0.000115,organic,-0.000037
5,speaker,0.000095,autonomous,-0.000036
6,advice,0.000073,everyone,-0.000034
7,references,0.000059,statute,-0.000033
8,crown,0.000059,chambers,-0.000028
9,senate,0.000053,organs,-0.000027


In [9]:
COMPS.PC0.sort_values(ascending=False).head(5)

term_str
art         0.000666
congress    0.000140
chamber     0.000075
deputies    0.000052
down        0.000039
Name: PC0, dtype: float64

In [10]:
COMPS.PC1.sort_values().head(5)

term_str
congress   -0.000221
peoples    -0.000054
chamber    -0.000047
deputies   -0.000047
organic    -0.000037
Name: PC1, dtype: float64

# Visualize


In [11]:
def vis_pcs(a, b, col="file_year"):
    x = f"PC{a}"
    y = f"PC{b}"
    fig = px.scatter(
        DCM,
        x,
        y,
        color=LIB[col],
        hover_name=DCM.index,
        hover_data={
            "title": LIB.title,
            "file_year": LIB.file_year,
            "n_chars": LIB.n_chars,
        },
        marginal_x="box",
        height=800,
        width=1000,
        title=f"Documents in {x} / {y} Space",
        template="plotly_white",
    )
    fig.update_traces(marker={"size": 9, "opacity": 0.8, "line": {"width": 0.5, "color": "#333333"}})
    fig.update_layout(legend_title_text=col, xaxis_title=x, yaxis_title=y)
    return fig

## PCA Visualization 1: Documents in first two components

In [12]:
vis_pcs(0, 1, col="original_year").show()

## PCA Visualization 1: loadings for first two components

In [13]:
def vis_loadings(a, b, label_top_n=20):
    x = f"PC{a}"
    y = f"PC{b}"
    loadings = COMPS[[x, y]].copy()
    loadings["term_str"] = loadings.index
    loadings["loading_strength"] = np.sqrt(loadings[x] ** 2 + loadings[y] ** 2)
    top_terms = loadings.loading_strength.nlargest(label_top_n).index
    loadings["label"] = np.where(loadings.index.isin(top_terms), loadings.index, "")

    fig = px.scatter(
        loadings,
        x=x,
        y=y,
        hover_name="term_str",
        text="label",
        height=800,
        width=1000,
        title=f"Term Loadings in {x} / {y} Space",
        template="plotly_white",
    )
    fig.update_traces(marker={"size": 7, "opacity": 0.55, "line": {"width": 0.4, "color": "#333333"}})
    fig.update_traces(textposition="top center")
    fig.update_layout(xaxis_title=x, yaxis_title=y)
    return fig

In [14]:
vis_loadings(0, 1).show()

## PCA Visualization 2: documents in second two components

In [15]:
vis_pcs(2, 3, col="original_year").show()

## PCA Visualization 2: loadings for second two components

In [16]:
vis_loadings(2, 3).show()

# Save


In [17]:
save_path = f"{output_dir}/{data_prefix}"

DCM.to_csv(f"{save_path}-PCA_DCM-{bag}.csv", sep=CSV_DELIM)
COMPS.to_csv(f"{save_path}-PCA_COMPS-{bag}.csv", sep=CSV_DELIM)
